In [25]:
import sys
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.utils as vutils
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [26]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

In [27]:
current_dir = Path.cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

In [33]:
transformation = transforms.Compose([transforms.ToTensor()])

In [34]:
train_dataset = datasets.MNIST(
    root=project_root / "data",
    train=True,
    transform=transformation,
    download=True
)

In [35]:
test_dataset = datasets.MNIST(
    root=project_root / "data",
    train=False,
    transform=transformation,
    download=True
)

In [36]:
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [39]:
x, y = next(iter(train_dataloader))

In [8]:
class Encoder(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        self.l_relu = nn.LeakyReLU()
        self.layer1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=4, padding=1, stride=2)
        self.batchnorm_1 = nn.BatchNorm2d(num_features=32)
        self.layer2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.batchnorm_2 = nn.BatchNorm2d(num_features=64)
        self.layer3 = nn.Conv2d(in_channels=64, out_channels=128, stride=2, padding=0, kernel_size=2)
        self.batchnorm_3 = nn.BatchNorm2d(num_features=128)
        # self.flatten = nn.Flatten()
        self.mu_head = nn.Linear(in_features=128*7*7, out_features=latent_dim)
        self.cov_head = nn.Linear(in_features=128*7*7, out_features=latent_dim)

    def forward(self, X):
        shape = X.shape
        X = self.layer1(X)              #(batch, 32, 14, 14)
        X = self.batchnorm_1(X)
        act1 = self.l_relu(X)
        X = self.layer2(act1)           #(batch, 64, 14, 14)
        X = self.batchnorm_2(X)
        act2 = self.l_relu(X)
        X = self.layer3(act2)           #(batch, 128, 7, 7)
        X = self.batchnorm_3(X)
        X = self.l_relu(X)
        X = X.view(shape[0], -1)        #(batch, 128*7*7)
        mu = self.mu_head(X)
        cov = self.cov_head(X)

        return mu, cov, act1, act2

In [9]:
class Decoder(nn.Module):
    def __init__(self, z_size: int):
        super().__init__()

        self.l_relu = nn.LeakyReLU()
        self.sigmoid = nn.Sigmoid()
        self.layer1 = nn.Linear(in_features=z_size, out_features=128*7*7)
        self.batchnorm_1 = nn.BatchNorm2d(num_features=128)
        self.layer2 = nn.ConvTranspose2d(in_channels=128, out_channels=64, stride=2, padding=1, kernel_size=3, output_padding=1)
        self.batchnorm_2 = nn.BatchNorm2d(num_features=64)
        self.layer3 = nn.ConvTranspose2d(in_channels=64, out_channels=32, stride=1, padding=1, kernel_size=3, output_padding=0)
        self.batchnorm_3 = nn.BatchNorm2d(num_features=32)
        self.layer4 = nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, padding=1, stride=2, output_padding=1)

    def forward(self, X):
        shape = X.shape
        X = self.layer1(X)
        X = X.view(shape[0], 128, 7, 7)
        X = self.batchnorm_1(X)
        X = self.l_relu(X)
        X = self.layer2(X)
        X = self.batchnorm_2(X)
        dact1 = self.l_relu(X)
        X = self.layer3(dact1)
        X = self.batchnorm_3(X)
        dact2 = self.l_relu(X)
        X = self.layer4(dact2)
        X = self.sigmoid(X)

        return X, dact1, dact2

In [41]:
def reparametarization_trick(mu, log_var):
    epsilon = torch.randn_like(mu)
    std = torch.exp(0.5*log_var)
    return mu + std * epsilon

In [24]:
def kl_divergence(mu, log_var):
    return -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())

In [10]:
# let's check if everythingis working correctly
latent_dim = 16
encoder = Encoder(latent_dim)
decoder = Decoder(latent_dim)

# test encoder
x = torch.randn(4, 1, 28, 28)
mu, cov, act1, act2 = encoder(x)
print("mu shape:  ", mu.shape)    # (4, 16)
print("act1 shape:", act1.shape)  # (4, 32, ?, ?)
print("act2 shape:", act2.shape)  # (4, 64, ?, ?)

# test decoder
z = torch.randn(4, latent_dim)
out, dact1, dact2 = decoder(z)
print("out shape: ", out.shape)   # (4, 1, 28, 28)
print("dact1 shape:", dact1.shape) # should match act2 channels
print("dact2 shape:", dact2.shape) # should match act1 channels

mu shape:   torch.Size([4, 16])
act1 shape: torch.Size([4, 32, 14, 14])
act2 shape: torch.Size([4, 64, 14, 14])
out shape:  torch.Size([4, 1, 28, 28])
dact1 shape: torch.Size([4, 64, 14, 14])
dact2 shape: torch.Size([4, 32, 14, 14])


### Implementing the Discriminators

In [14]:
class Discriminator1(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.l_relu = nn.LeakyReLU()
        self.sigmoid = nn.Sigmoid()
        self.layer1 = nn.Conv2d(in_channels=64, out_channels=128, stride=2, padding=0, kernel_size=2)
        self.batchnorm1 = nn.BatchNorm2d(num_features=128)
        self.flatten = nn.Flatten()
        self.head = nn.Linear(in_features=128*7*7, out_features=1)

    def forward(self, X):
        X = self.layer1(X)
        X = self.batchnorm1(X)
        X = self.l_relu(X)
        X = self.flatten(X)
        X = self.head(X)
        X = self.sigmoid(X)
        
        return X

In [15]:
class Discriminator2(nn.Module):
    def __init__(self):
        super().__init__()
        self.l_relu = nn.LeakyReLU()
        self.sigmoid = nn.Sigmoid()
        self.layer1 = nn.Conv2d(in_channels=32, out_channels=64, stride=2, padding=0, kernel_size=2)
        self.batchnorm1 = nn.BatchNorm2d(num_features=64)
        self.flatten = nn.Flatten()
        self.head = nn.Linear(in_features=64*7*7, out_features=1)
        
    def forward(self, X):
        X = self.layer1(X)
        X = self.batchnorm1(X)
        X = self.l_relu(X)
        X = self.flatten(X)
        X = self.head(X)
        X = self.sigmoid(X)
        
        return X

In [16]:
# testing the discriminators
d1 = Discriminator1()
d2 = Discriminator2()

# simulate real/fake inputs
act2  = torch.randn(4, 64, 14, 14)  # encoder act2 (real for d1)
dact1 = torch.randn(4, 64, 14, 14)  # decoder dact1 (fake for d1)
act1  = torch.randn(4, 32, 14, 14)  # encoder act1 (real for d2)
dact2 = torch.randn(4, 32, 14, 14)  # decoder dact2 (fake for d2)

print(d1(act2).shape)   # should be (4, 1)
print(d1(dact1).shape)  # should be (4, 1)
print(d2(act1).shape)   # should be (4, 1)
print(d2(dact2).shape)  # should be (4, 1)

torch.Size([4, 1])
torch.Size([4, 1])
torch.Size([4, 1])
torch.Size([4, 1])


In [43]:
# writing the training function
def train_the_archi(
    dataloader: DataLoader,
    encoder: Encoder,
    decoder: Decoder,
    d1: Discriminator1,
    d2: Discriminator2,
    vae_optimizers: optim.Adam,
    decoder_optimizer: optim.Adam,
    d1_optimizer: optim.Adam,
    d2_optimizer: optim.Adam,
    bce_loss: nn.BCELoss,
    device: torch.device,
    epochs: int,
):
    total_vae_cost = []
    total_d1_cost = []
    total_d2_cost = []
    total_dd_cost = []
    for epoch in range(epochs):
        total_vae_loss = 0
        total_d1_loss = 0
        total_d2_loss = 0
        total_dd_loss = 0
        for batch, _ in dataloader:
            batch = batch.to(device)
            # Encoder
            mu, cov, act1, act2 = encoder(batch)
            
            # reparametarization trick
            latent_space = reparametarization_trick(mu=mu, log_var=cov)
            
            # decoder
            generated_img, dact1, dact2 = decoder(latent_space)
            
            # calculating the discrimators losses - step-1
            batch_size = batch.shape[0]
            ones = torch.ones(batch_size, 1).to(device=device)
            zeros = torch.zeros(batch_size, 1).to(device=device)
            d1_out = d1(dact1.detach())
            d2_out = d2(dact2.detach())
            d1_loss = bce_loss(d1(act2.detach()), ones) + bce_loss(d1_out, zeros)
            d2_loss = bce_loss(d2(act1.detach()), ones) + bce_loss(d2_out, zeros)
            total_d1_loss += d1_loss.item()
            total_d2_loss += d2_loss.item()
            
            # updating the discriminators
            d1_optimizer.zero_grad()
            d1_loss.backward()
            d1_optimizer.step()
            
            d2_optimizer.zero_grad()
            d2_loss.backward()
            d2_optimizer.step()
            
            # calculating the vae loss - step-2
            mu, cov, _, _ = encoder(batch)
            latent_space = reparametarization_trick(mu, cov)
            generated_img, _, _ = decoder(latent_space)
            kldiv_loss = kl_divergence(mu=mu, log_var=cov)
            recons_loss = bce_loss(generated_img, batch)
            vae_loss = kldiv_loss + recons_loss
            total_vae_loss += vae_loss.item()
            
            # updating the weights of the encoder and decoder
            vae_optimizers.zero_grad()
            vae_loss.backward()
            vae_optimizers.step()
            
            # calculating the loss of the decoder based on the discrminator - step-3
            mu, cov, _, _ = encoder(batch)
            latent_space = reparametarization_trick(mu.detach(), cov.detach())
            _, dact1_new, dact2_new = decoder(latent_space)
            dd_loss = bce_loss(d1(dact1_new), ones) + bce_loss(d2(dact2_new), ones)
            total_dd_loss += dd_loss.item()
            
            # updating the decoders weights
            decoder_optimizer.zero_grad()
            dd_loss.backward()
            decoder_optimizer.step()
            
        avg_d1_loss = total_d1_loss/len(dataloader)
        avg_d2_loss = total_d2_loss/len(dataloader)
        avg_vae_loss = total_vae_loss/len(dataloader)
        avg_dd_loss = total_dd_loss/len(dataloader)
        
        total_d1_cost.append(avg_d1_loss)
        total_d2_cost.append(avg_d2_loss)
        total_vae_cost.append(avg_vae_loss)
        total_dd_cost.append(avg_dd_loss)
        
        print(f"Epoch {epoch+1}/{epochs} | "
          f"VAE: {avg_vae_loss:.2f} | "
          f"D1: {avg_d1_loss:.2f} | "
          f"D2: {avg_d2_loss:.2f} | "
          f"DD: {avg_dd_loss:.2f}")
        
    return total_d1_cost, total_d2_cost, total_dd_cost, total_vae_cost

In [44]:
# let's create all of the optimizers and the components
LATENT_DIM = 16
encoder = Encoder(latent_dim=LATENT_DIM).to(device=device)
decoder = Decoder(z_size=LATENT_DIM).to(device=device)
d1 = Discriminator1().to(device=device)
d2 = Discriminator2().to(device=device)

lr = 1e-4
vae_optimizers = optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=lr
)

decoder_optimizer = optim.Adam(
    decoder.parameters(), lr=lr
)

d1_optimizer = optim.Adam(d1.parameters(), lr=lr)
d2_optimizer = optim.Adam(d2.parameters(), lr=lr)

In [45]:
# create some loss functions
bce_loss = nn.BCELoss()

In [46]:
total_d1_cost, total_d2_cost, total_dd_cost, total_vae_cost = train_the_archi(
    dataloader=train_dataloader,
    encoder=encoder,
    decoder=decoder,
    d1=d1,
    d2=d2,
    vae_optimizers=vae_optimizers,
    decoder_optimizer=decoder_optimizer,
    d1_optimizer=d1_optimizer,
    d2_optimizer=d2_optimizer,
    bce_loss=bce_loss,
    device=device,
    epochs=1
)

Epoch 1/1 | VAE: 18.12 | D1: 0.10 | D2: 0.35 | DD: 6.06
